In [0]:
%sql
CREATE CATALOG imdb_catalog

In [0]:
%sql
CREATE SCHEMA imdb_catalog.imdb_schema

In [0]:
%sql
CREATE VOLUME imdb_catalog.imdb_schema.imdb_volume

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.appName("imdb_project").getOrCreate()

In [0]:
imdb_raw_df = spark.read.format('csv').option('header', 'true').option('inferSchema','true').load("/Volumes/imdb_catalog/imdb_schema/imdb_volume/tmdb_movies.csv")

In [0]:
imdb_raw_df.display()

id,title,genre,release_date,release_year,budget,revenue,runtime,vote_average,vote_count,production_country
1,Movie Title 1,"Science Fiction, Comedy, Horror",1957-01-01,1957,36972721,1.3069283983126548E8,123,4.496790402095412,9221,Germany
2,Movie Title 2,"Documentary, Romance",1999-01-01,1999,96359319,2.947974958574514E8,146,5.803305921613626,4592,China
3,Movie Title 3,"Science Fiction, Action",1941-01-01,1941,135172742,1.0484089374112095E8,168,2.710639127014427,7146,Japan
4,Movie Title 4,"Comedy, Drama, Adventure",1961-01-01,1961,129049379,1.5355849261335883E8,134,5.6313980592814765,8363,Germany
5,Movie Title 5,"Comedy, Romance",1969-01-01,1969,151390401,1.008290657565172E8,116,4.24586189743736,9487,Japan
6,Movie Title 6,Adventure,2020-01-01,2020,169912312,5.724922972839584E8,163,1.167284903613779,3801,India
7,Movie Title 7,Documentary,1962-01-01,1962,47113615,2.3094190929998395E8,174,3.5756880694301922,6228,India
8,Movie Title 8,"Comedy, Romance",1960-01-01,1960,42844528,8.601494050691138E7,150,8.112310080567294,9485,India
9,Movie Title 9,"Horror, Science Fiction, Documentary",2007-01-01,2007,96564462,4.945880339794745E7,149,5.886860615538395,8356,United Kingdom
10,Movie Title 10,"Drama, Documentary",1992-01-01,1992,14771093,6.353765276033808E7,107,6.632066395432246,3680,China


In [0]:
imdb_raw_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- budget: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: integer (nullable = true)
 |-- production_country: string (nullable = true)



In [0]:
imdb_df =imdb_raw_df.fillna({
    "title": 'NA',
    "genre": 'NA',
    "release_year": 2000,
    "budget": 0,
    "revenue": 0,
    "runtime": 0,
    "vote_count": 0,
    "vote_average": 0.0,
    "production_country": 'NA'
})

imdb_df = imdb_df.withColumn('release_date',
                             coalesce(col('release_date'), to_date(lit('2000-01-01'))))

In [0]:
imdb_df.dropDuplicates(['id'])

DataFrame[id: int, title: string, genre: string, release_date: date, release_year: int, budget: int, revenue: double, runtime: int, vote_average: double, vote_count: int, production_country: string]

In [0]:
imdb_df = imdb_df.withColumns({'revenue': round(col('revenue'),2),
                             'vote_average': round(col('vote_average'),2)
                             })

In [0]:
imdb_df = imdb_df.withColumn('profit', round((col('revenue') - col('budget')),2))

In [0]:
imdb_df = imdb_df.withColumn('revenue_category',
                             when(col('revenue') > 100000000, "Blockbuster")
                             .when(col('revenue') > 50000000, "Hit")
                             .when(col('revenue') > 10000000, "Average")
                             .otherwise('Flop')
                             )


In [0]:
imdb_df = imdb_df.withColumns({
    'movie_age': current_date() - to_date(col("release_date")),
    'movie_years': year(current_date()) - col("release_year")
    })

In [0]:
genre_df = imdb_df.select('genre').distinct()

joined_df = imdb_df.join(genre_df, 'genre', 'left')

In [0]:
joined_df.display()

genre,id,title,release_date,release_year,budget,revenue,runtime,vote_average,vote_count,production_country,profit,revenue_category,movie_age,movie_years
"Science Fiction, Comedy, Horror",1,Movie Title 1,1957-01-01,1957,36972721,1.3069283983E8,123,4.5,9221,Germany,9.372011883E7,Blockbuster,INTERVAL '25299' DAY,69
"Documentary, Romance",2,Movie Title 2,1999-01-01,1999,96359319,2.9479749586E8,146,5.8,4592,China,1.9843817686E8,Blockbuster,INTERVAL '9959' DAY,27
"Science Fiction, Action",3,Movie Title 3,1941-01-01,1941,135172742,1.0484089374E8,168,2.71,7146,Japan,-3.033184826E7,Blockbuster,INTERVAL '31143' DAY,85
"Comedy, Drama, Adventure",4,Movie Title 4,1961-01-01,1961,129049379,1.5355849261E8,134,5.63,8363,Germany,2.450911361E7,Blockbuster,INTERVAL '23838' DAY,65
"Comedy, Romance",5,Movie Title 5,1969-01-01,1969,151390401,1.0082906576E8,116,4.25,9487,Japan,-5.056133524E7,Blockbuster,INTERVAL '20916' DAY,57
Adventure,6,Movie Title 6,2020-01-01,2020,169912312,5.7249229728E8,163,1.17,3801,India,4.0257998528E8,Blockbuster,INTERVAL '2289' DAY,6
Documentary,7,Movie Title 7,1962-01-01,1962,47113615,2.309419093E8,174,3.58,6228,India,1.838282943E8,Blockbuster,INTERVAL '23473' DAY,64
"Comedy, Romance",8,Movie Title 8,1960-01-01,1960,42844528,8.601494051E7,150,8.11,9485,India,4.317041251E7,Hit,INTERVAL '24204' DAY,66
"Horror, Science Fiction, Documentary",9,Movie Title 9,2007-01-01,2007,96564462,4.94588034E7,149,5.89,8356,United Kingdom,-4.71056586E7,Average,INTERVAL '7037' DAY,19
"Drama, Documentary",10,Movie Title 10,1992-01-01,1992,14771093,6.353765276E7,107,6.63,3680,China,4.876655976E7,Hit,INTERVAL '12516' DAY,34


In [0]:
top_genre_by_rev = imdb_df.groupBy('genre')\
.sum('revenue')\
.orderBy("sum(revenue)", ascending=False)

In [0]:
top_genre_by_rev.display()

genre,sum(revenue)
Adventure,2.77794168453E9
Horror,2.22683871082E9
Documentary,1.60275023849E9
Romance,1.19665315226E9
Action,1.14392430356E9
"Romance, Documentary, Horror",1.10500010131E9
"Comedy, Drama, Science Fiction",9.7091110214E8
"Drama, Horror, Science Fiction",8.5776056015E8
"Adventure, Drama",8.5203277247E8
Drama,8.363156043E8


In [0]:
s1 = imdb_df.groupBy('production_country').count()

In [0]:
s1.display()

production_country,count
United Kingdom,13
Japan,10
Canada,16
United States of America,11
China,13
India,14
Germany,9
France,14


In [0]:
avg_rating = imdb_df.groupBy('genre').avg('vote_average')
avg_rating.display()
avg_runtime = imdb_df.groupBy('genre').avg('runtime')
avg_runtime.display()
avg_profit = imdb_df.groupBy('genre').avg('profit')
avg_profit.display()

genre,avg(vote_average)
"Comedy, Drama, Adventure",5.63
"Drama, Comedy",5.734999999999999
Horror,6.8100000000000005
"Action, Romance",5.35
"Drama, Action",5.485
"Drama, Horror",5.47
"Science Fiction, Comedy",8.16
"Documentary, Action",2.88
"Drama, Documentary, Comedy",1.09
"Action, Adventure, Horror",7.81


genre,avg(runtime)
"Comedy, Drama, Adventure",134.0
"Drama, Comedy",117.0
Horror,135.2
"Action, Romance",99.0
"Drama, Action",140.0
"Drama, Horror",131.0
"Science Fiction, Comedy",83.0
"Documentary, Action",121.0
"Drama, Documentary, Comedy",82.0
"Action, Adventure, Horror",107.0


genre,avg(profit)
"Comedy, Drama, Adventure",2.450911361E7
"Drama, Comedy",1.7872415107E8
Horror,3.0452819596400005E8
"Action, Romance",-1.1367453975E8
"Drama, Action",1.2458692703E8
"Drama, Horror",3.1099577666E8
"Science Fiction, Comedy",-5.58025972E7
"Documentary, Action",3.591565097E7
"Drama, Documentary, Comedy",1.6777459567E8
"Action, Adventure, Horror",5.492961472E7


In [0]:
df1 = imdb_df.repartition('genre')

In [0]:
cleaned_df = imdb_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in imdb_df.columns
])

In [0]:
cleaned_df.display()

id,title,genre,release_date,release_year,budget,revenue,runtime,vote_average,vote_count,production_country,profit,revenue_category,movie_age,movie_years
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
window_spec = Window.partitionBy('genre').orderBy(col('revenue').desc())

imdb_df.withColumn('top_movie', row_number().over(window_spec)).filter(col('top_movie') == 1)

DataFrame[id: int, title: string, genre: string, release_date: date, release_year: int, budget: int, revenue: double, runtime: int, vote_average: double, vote_count: int, production_country: string, profit: double, revenue_category: string, movie_age: interval day, movie_years: int, top_movie: int]

In [0]:
display(imdb_df.limit(3))

id,title,genre,release_date,release_year,budget,revenue,runtime,vote_average,vote_count,production_country,profit,revenue_category,movie_age,movie_years
1,Movie Title 1,"Science Fiction, Comedy, Horror",1957-01-01,1957,36972721,1.3069283983E8,123,4.5,9221,Germany,9.372011883E7,Blockbuster,INTERVAL '25299' DAY,69
2,Movie Title 2,"Documentary, Romance",1999-01-01,1999,96359319,2.9479749586E8,146,5.8,4592,China,1.9843817686E8,Blockbuster,INTERVAL '9959' DAY,27
3,Movie Title 3,"Science Fiction, Action",1941-01-01,1941,135172742,1.0484089374E8,168,2.71,7146,Japan,-3.033184826E7,Blockbuster,INTERVAL '31143' DAY,85


In [0]:
final_df = imdb_df.select(
    'id',
    'title',
    'genre',
    'release_year',
    'revenue',
    'vote_average',
    'production_country',
    'profit',
    'revenue_category'
)

In [0]:
final_df.display()

id,title,genre,release_year,revenue,vote_average,production_country,profit,revenue_category
1,Movie Title 1,"Science Fiction, Comedy, Horror",1957,1.3069283983E8,4.5,Germany,9.372011883E7,Blockbuster
2,Movie Title 2,"Documentary, Romance",1999,2.9479749586E8,5.8,China,1.9843817686E8,Blockbuster
3,Movie Title 3,"Science Fiction, Action",1941,1.0484089374E8,2.71,Japan,-3.033184826E7,Blockbuster
4,Movie Title 4,"Comedy, Drama, Adventure",1961,1.5355849261E8,5.63,Germany,2.450911361E7,Blockbuster
5,Movie Title 5,"Comedy, Romance",1969,1.0082906576E8,4.25,Japan,-5.056133524E7,Blockbuster
6,Movie Title 6,Adventure,2020,5.7249229728E8,1.17,India,4.0257998528E8,Blockbuster
7,Movie Title 7,Documentary,1962,2.309419093E8,3.58,India,1.838282943E8,Blockbuster
8,Movie Title 8,"Comedy, Romance",1960,8.601494051E7,8.11,India,4.317041251E7,Hit
9,Movie Title 9,"Horror, Science Fiction, Documentary",2007,4.94588034E7,5.89,United Kingdom,-4.71056586E7,Average
10,Movie Title 10,"Drama, Documentary",1992,6.353765276E7,6.63,China,4.876655976E7,Hit


In [0]:
final_df.write.mode('overwrite').parquet('/Volumes/imdb_catalog/imdb_schema/imdb_volume/final_clean_movies')